# MAS-SHT v12.0 — Kaggle Experiment Harness

**Before running:**
1. Session options (⚙️ right panel) → Accelerator → **GPU T4 x1**
2. Add Kaggle Secrets (right panel → Add Notebook Secret):
   - `GITHUB_TOKEN` — GitHub PAT with "Contents: read" (skip if repo is public)
   - `HF_API_KEY` — HuggingFace token (needed to download Qwen model)
3. Run cells top to bottom. Re-running Cell 3 resumes from checkpoint.

In [ ]:
# === Cell 1 — Kaggle Setup ====================================================
# Installs deps, clones private GitHub repo via PAT, loads Kaggle Secrets.
# Re-running is safe: git pull updates the repo, pip skips installed pkgs.
#
# REQUIRED Kaggle Secrets (Add Notebook Secret in right-side panel):
#   GITHUB_TOKEN  — GitHub PAT with "Contents: read" permission
#                   (needed only if repo is private; skip if repo is public)
#   HF_API_KEY    — HuggingFace token (only for local-model runs; not needed for Groq)
#   GROQ_API_KEY  — Groq API key (REQUIRED for the Qwen3-32B Groq run)
#
# GPU: Session options → Accelerator → GPU T4 x1  (must enable before running)

import os, sys, subprocess

DEPS = [
    'openai>=1.40.0', 'sympy', 'statsmodels',
    'bitsandbytes',  # [v10.3] 4-bit quant for 7B models
    'scipy', 'pandas', 'matplotlib', 'tqdm', 'datasets',
    'transformers>=4.44.0', 'accelerate', 'huggingface_hub',
    'python-dotenv', 'tabulate', 'requests',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + DEPS)
print("Deps installed.")

# ── Kaggle Secrets ────────────────────────────────────────────────────────────
def _get_secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.environ.get(name)

_gh_token = _get_secret('GITHUB_TOKEN')   # None → assumes repo is public
_hf_token = _get_secret('HF_API_KEY')
_groq_token = _get_secret('GROQ_API_KEY')

if _hf_token:
    os.environ['HF_API_KEY']        = _hf_token
    os.environ['HUGGINGFACE_TOKEN'] = _hf_token
    os.environ['HF_TOKEN']          = _hf_token
    print("HF token loaded.")
else:
    print("WARNING: HF_API_KEY not set — model download may fail for gated repos.")

if _groq_token:
    os.environ['GROQ_API_KEY'] = _groq_token
    print("Groq token loaded.")
else:
    print("WARNING: GROQ_API_KEY not set — the Qwen3-32B Groq run WILL FAIL. Add it in Kaggle Secrets.")

# ── Clone / update repo ───────────────────────────────────────────────────────
REPO_DIR  = '/kaggle/working/llm_thesis'
REPO_NAME = 'marios4371/llm_thesis'

if _gh_token:
    REPO_URL = f'https://{_gh_token}@github.com/{REPO_NAME}.git'
else:
    REPO_URL = f'https://github.com/{REPO_NAME}.git'

if not os.path.isdir(REPO_DIR):
    print("Cloning repo…")
    result = subprocess.run(
        ['git', 'clone', REPO_URL, REPO_DIR],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        # Hide token from error message
        err = result.stderr.replace(_gh_token or '', '***') if _gh_token else result.stderr
        raise RuntimeError(f"git clone failed:\n{err}")
    print("Clone OK.")
else:
    print("Repo already present — pulling latest…")
    # Set the remote URL (in case token changed)
    subprocess.run(['git', '-C', REPO_DIR, 'remote', 'set-url', 'origin', REPO_URL],
                   capture_output=True)
    result = subprocess.run(
        ['git', '-C', REPO_DIR, 'pull', '--ff-only'],
        capture_output=True, text=True
    )
    print(result.stdout.strip() or "Already up to date.")

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# ── Output directories ────────────────────────────────────────────────────────
ROOT          = '/kaggle/working/MAS_SHT'
RESULTS_DIR   = f'{ROOT}/results'
CHECKPOINT_DIR= f'{ROOT}/checkpoints'
ARTIFACTS_DIR = f'{ROOT}/artifacts'
for d in [RESULTS_DIR, CHECKPOINT_DIR, ARTIFACTS_DIR]:
    os.makedirs(d, exist_ok=True)

# ── GPU check ─────────────────────────────────────────────────────────────────
import torch
print(f"\nGPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected! Inference will be very slow on CPU.")
    print("Go to: Session options → Accelerator → GPU T4 x1, then restart.")

print(f"\nRepo  : {REPO_DIR}")
print(f"Results: {RESULTS_DIR}")
print(f"Checkpts: {CHECKPOINT_DIR}")


## Cell 2 — Dataset

In [ ]:
# === Cell 2 — Dataset ==========================================================
# Uses EnhancedProblemManager — supports 17 benchmarks.
# PUBLISHABLE TARGET: n=120, seed=42, 4 benchmarks  (~15h on T4 GPU).
#
# Grade-school level (literature reference):
#   gsm8k_test        GSM8K test            (★,    1319 problems)
#   gsm-hard          larger numbers        (★★,   1319 problems)
#   svamp             structural variation  (★★,   1000 problems)
#   gsm-symbolic-p2   +2 extra clauses      (★★★, ~5000 problems)
#   gsm-plus          8 perturbation types  (★★,  ~3000 problems)
#   math500           Hendrycks competition (★★★★)
#
# Uncontaminated 2025/2026 benchmarks (NEW — main thesis contribution):
#   aime_2026         AIME 2026        (★★★★★, 30 probs, integer answers, freshest)
#   aime_2025         AIME 2025        (★★★★★, 30 probs)
#   hmmt_feb_2026     HMMT Feb 2026    (★★★★★, ~35 probs, harder than AIME)
#   hmmt_feb_2025     HMMT Feb 2025    (★★★★★, ~35 probs)
#   olymmath_hard     OlymMATH EN-HARD (★★★★★, frontier ~58%)
#   amo_bench         AMO-Bench        (★★★★★, 50 original IMO problems)
#   omni_math         Omni-MATH        (★★★★★, 4428 olympiad problems)
#   livemathbench     LiveMathBench    (★★★★,  anti-contamination, monthly updates)

import sys
import random
sys.path.insert(0, REPO_DIR)
import importlib
for _m in ['Mas_solver']:
    if _m in sys.modules: del sys.modules[_m]
import Mas_solver
from Mas_solver import EnhancedProblemManager

# ── Choose datasets and sample size ─────────────────────────────────────────────
# [v10.6] Difficulty mix chosen so MAS deltas are MEASURABLE on a 7B model:
#   gsm8k_test : grade-school, near-ceiling — proves MAS pipeline works
#   gsm-hard   : larger numbers, arithmetic-heavy — where SIV/code help most
#   svamp      : structural variation — tests blueprint robustness
# (math500/aime: floor effects at 7B scale hide SIV delta; exclude here)
#
# [v12.2] UNEVEN per-dataset draw (changed from the flat 50/50/50 split used in
# the n=150 pilot / v12.0-v12.1 runs). Measured genuine-blueprint yield
# (>=2 numeric givens, i.e. SIV Layer 2 actually has multiple variables to
# reason about) on that run: gsm-hard 26% (13/50), gsm8k_test 22% (11/50),
# svamp 10% (5/50). SIV's evaluation is starved by low coverage (only 29/150
# genuine cases total), so we now weight the draw toward gsm-hard and away
# from svamp to raise the genuine-SIV sample size at the SAME total N — no
# model/config change, zero extra GPU cost per problem. svamp is kept (not
# dropped) so the existing per-benchmark comparison table still has all three.
#
# IMPORTANT — this produces a DIFFERENT problem set than the n=150 pilot and
# the v12.0/v12.1 runs (those used the flat 50/50/50 draw). Before the FIRST
# run under this new mix, reset the stale checkpoints so old- and new-mix
# problems don't get silently concatenated into the same CSV:
#   RESET_SYSTEMS = ['b1_direct', 'b2_cot', 'b_pal', 'mas_sht_math7b']
# in Cell 3 for that one commit, then set it back to [] to resume normally.
# (Your already-downloaded pilot/v12.0/v12.1 CSVs are untouched — this only
# clears the *Kaggle-side* in-progress checkpoint.)
DATASET_WEIGHTS = [
    ('gsm-hard',   75),   # 50% — highest genuine-blueprint yield (26%)
    ('gsm8k_test', 50),   # 33% — middle yield (22%), unchanged in absolute count
    ('svamp',      25),   # 17% — lowest yield (10%), reduced but not dropped
]
N_PROBLEMS = sum(n for _, n in DATASET_WEIGHTS)   # 150, same total as before
DATASETS   = [name for name, _ in DATASET_WEIGHTS]
SEED       = 42               # for reproducibility — change to 43, 44 for multi-seed

manager  = EnhancedProblemManager(random_seed=SEED)
raw      = []
for _ds_name, _count in DATASET_WEIGHTS:
    # One dataset per call -> load_random_problems draws exactly `_count` from
    # THAT dataset only (per_ds = _count when len(datasets_list) == 1).
    raw.extend(manager.load_random_problems([_ds_name], _count))

# [v12.2 FIX] Shuffle the combined list. `random` here is the SAME module
# object EnhancedProblemManager.__init__ already seeded (random.seed(SEED)
# inside Mas_solver.py -- there is only one 'random' module per process), so
# this stays fully deterministic/reproducible. Without this, `raw` is BLOCKED
# by dataset (all 75 gsm-hard, then all 50 gsm8k, then all 25 svamp) in the
# exact order DATASET_WEIGHTS lists them. gsm-hard is both first AND by far
# the slowest under the full MAS pipeline (~233s/problem average, some over
# 600s) -- so a run that gets cut short (Kaggle GPU session limit, manual
# stop, disconnect) ends up with 100% gsm-hard and 0% gsm8k/svamp, which is
# exactly what happened to the b_pal and mas_sht_math7b runs on 2026-07-06
# (both stopped at 75/150, precisely the end of the gsm-hard block). This
# shuffle does NOT change which 150 problems are drawn, only their
# processing order, so it's safe to apply on top of an existing checkpoint --
# already-completed problem_ids are still recognized and skipped on resume.
random.shuffle(raw)

def _parse_answer(a):
    """Extract numeric answer. Handles GSM8K '#### N' format."""
    if a is None: return None
    s = str(a)
    if '####' in s:
        s = s.split('####')[-1].strip()
    s = s.replace(',', '')
    try:    return float(s)
    except: return s

PROBLEMS = [
    {
        'problem_id':  p['id'],
        'question':    p['puzzle'],
        'gold_answer': _parse_answer(p.get('answer')),
        'dataset':     p.get('dataset', 'unknown'),
    }
    for p in raw if p.get('answer') is not None
]

# [v10.6] Keep only numerically-gradable items (gold parses to float).
# A few MATH500 golds are symbolic expressions; the numeric grader cannot
# score them, deflating every system equally. Excluding them upfront keeps
# the accuracy comparison clean. AIME/GSM golds are always numeric.
_n0 = len(PROBLEMS)
PROBLEMS = [p for p in PROBLEMS if isinstance(p['gold_answer'], float)]
if _n0 - len(PROBLEMS):
    print(f'Numeric-gold filter: dropped {_n0 - len(PROBLEMS)} symbolic-gold item(s), kept {len(PROBLEMS)}')

from collections import Counter
print(f'Loaded {len(PROBLEMS)} problems  (seed={SEED})')
for ds, cnt in Counter(p['dataset'] for p in PROBLEMS).items():
    print(f'  {ds}: {cnt}')
print('Example:', PROBLEMS[0]['question'][:100], '...')


## Cell 2.5 — Pre-flight check (~10–15 min)

**Run this BEFORE Cell 3.** It solves 2 problems end-to-end with the 7B model and verifies
the Mathematician actually produces JSON blueprints (the 1.5B could not). Costs ~15 min,
saves a wasted 6-hour run. Must end with **`PRE-FLIGHT: PASS`**.

In [ ]:
# === Cell 2.5 — PRE-FLIGHT CHECK ===============================================
# Verifies the full MAS pipeline (blueprint -> programmer -> SIV) works with the
# 7B model BEFORE committing 6-8 GPU-hours. If it prints FAIL, do NOT run Cell 3
# — copy the log output (especially WARNING lines) back to Claude.

import time, gc, sys
sys.path.insert(0, REPO_DIR)
for _m in ['Mas_solver']:
    if _m in sys.modules: del sys.modules[_m]
import Mas_solver
from Mas_solver import QualityAwarePipeline, _extract_last_number

# [v13.0] Toggle: mixed-by-output-contract preset -- Instruct on every role
# whose output must be PARSED (Mathematician/JSON blueprint, Hypothesis-
# Generator/JSON alternatives, Judge/constrained tag), Math-7B on the roles
# graded on raw math (Baseline CoT anchor, Programmer). Flip to False for a
# single homogeneous model on every role. Cell 3 reuses this SAME variable,
# so pre-flight always tests what actually runs.
USE_MIXED_PRESET = True   # False -> homogeneous 'qwen_math_7b_local' for all roles
MAS_PRESET = ('qwen_math7b_mixed' if USE_MIXED_PRESET
              else 'qwen_math_7b_local')
assert MAS_PRESET in Mas_solver.HETEROGENEOUS_PRESETS, \
    f'{MAS_PRESET!r} not found — Cell 1 (git pull) was not run with the latest code!'

pf_pipeline = QualityAwarePipeline(
    heterogeneous_preset=MAS_PRESET, use_cache=False,
    enable_siv=True, enable_sht=True,
    evaluation_mode=True, dataset_seed=42,
)

# Test one short and one long problem from the loaded set
_by_len  = sorted(PROBLEMS, key=lambda p: len(p['question']))
pf_tests = [_by_len[0], _by_len[-1]]

pf_times, pf_ok, pf_bps = [], [], []
for tp in pf_tests:
    print(f"\n--- pre-flight: {tp['problem_id']} ({tp['dataset']}, {len(tp['question'])} chars)")
    t0 = time.time()
    out = pf_pipeline.solver.solve(tp['question'], str(tp['gold_answer']))
    dt = time.time() - t0
    ans = out.get('mas', {}).get('answer')
    bp  = out.get('siv', {}).get('blueprint_answer')
    fb  = out.get('mas', {}).get('used_baseline_fallback') or out.get('mas', {}).get('local_hf_fallback')
    print(f"    answer={ans!r}   gold={tp['gold_answer']}   blueprint_answer={bp!r}")
    print(f"    fallback_used={bool(fb)}   time={dt:.0f}s")
    # [v10.7] PASS = pipeline produced a numeric final answer. Blueprint may be
    # empty: Qwen2.5-Math emits chain-of-thought, not JSON, so MAS runs via the
    # CoT->SymPy fallback path. That is a valid (if degraded) mode, not a failure.
    pf_ok.append(_extract_last_number(str(ans)) is not None)
    if bp is None:
        print('    NOTE: blueprint empty -> running in CoT-fallback mode (SIV limited)')
    pf_bps.append(bp)
    pf_times.append(dt)

# Free the 7B before Cell 3 re-loads it (avoid double-resident 14GB models)
for _k, _c in list(getattr(pf_pipeline, '_client_cache', {}).items()):
    if getattr(_c, '_local_model', None) is not None:
        _c._local_model = None
        _c._local_tokenizer = None
del pf_pipeline
gc.collect()
try:
    import torch; torch.cuda.empty_cache()
except Exception:
    pass

# [v13.0] Mathematician-liveness guard. On 2026-07-12 a T4 x2 session OOM'd
# loading the SECOND model (the Instruct Mathematician): every blueprint died
# with ERROR_GENERATION, yet pre-flight still PASSed on the numeric-answer
# criterion and an 8h commit would have run entirely in fallback mode. Under
# the mixed preset, zero formed blueprints across the test problems means the
# Mathematician is dead -- fail loudly instead.
if USE_MIXED_PRESET and all(b is None for b in pf_bps):
    pf_ok.append(False)
    print('FAIL-GUARD: mixed preset, but NO blueprint formed on any test problem.')
    print('  The Instruct Mathematician most likely failed to load (look for')
    print('  "load FAILED (OutOfMemoryError" above). Do NOT run Cell 3.')

avg = sum(pf_times) / len(pf_times)
eta_h = len(PROBLEMS) * (avg + 3 * 70) / 3600   # MAS + 3 baseline calls (~70s each)
print('\n' + '=' * 62)
if all(pf_ok):
    print(f'PRE-FLIGHT: PASS   (avg MAS solve {avg:.0f}s/problem)')
    print(f'Projected full run: ~{eta_h:.1f} h  ({len(PROBLEMS)} problems x 4 systems)')
    print('-> Proceed to Cell 3.')
else:
    print('PRE-FLIGHT: FAIL — blueprint pipeline is not working with this model.')
    print('-> Do NOT run Cell 3. Send the log above to Claude for diagnosis.')
print('=' * 62)

# [v14.0] This used to be print-only. On 2026-07-31 a mas_full commit ran to
# completion for ~8h with the Mathematician erroring out on EVERY single one of
# 150 problems (blueprint_givens={} in every row -- functionally a baseline-only
# run) even though this exact guard almost certainly printed FAIL above: a
# "Save & Run All (Commit)" does not stop at a printed warning, only at an
# uncaught exception. Raising here is what actually prevents the next 8h from
# silently repeating that.
if not all(pf_ok):
    raise RuntimeError(
        'Pre-flight FAILED (see PRE-FLIGHT/FAIL-GUARD output above). Aborting '
        'before Cell 3 to avoid burning the GPU-hour budget on a broken '
        'pipeline. Fix the underlying issue (most likely the Mathematician '
        'erroring on every call -- look for OutOfMemoryError/ERROR_GENERATION '
        'in the log, or a GPU accelerator mismatch) and re-run from Cell 1.'
    )


## Cell 3 — Experiment Runner

Runs **B1/B2/B4/B₀ₚₐₗ/Bₚₒₜ** baselines + MAS-SHT. Safe to interrupt and resume.  

In [ ]:
# === Cell 3 — Experiment Runner ================================================
# Runs all baselines (B1, B2, B4, B_PAL, B_PoT) + MAS-SHT in one pass.
# Checkpointing every 5 problems → safe to interrupt and resume.
#
# HOW TO RESUME:
#   Just re-run this cell. Already-completed problems are skipped automatically.
#
# HOW TO RESET (fresh run):
#   Set RESET_SYSTEMS to the system names you want to restart from scratch,
#   e.g. RESET_SYSTEMS = ['b1_direct', 'b2_cot', 'b_pal', 'mas_sht_math7b']  # [v12.2] ONE-TIME:
# Cell 2's dataset mix changed (weighted toward gsm-hard) -> old checkpoints under these
# names hold the STALE 50/50/50-mix problems and would get silently concatenated with the
# new mix otherwise. After this first commit under the new mix completes, set this back to
# RESET_SYSTEMS = [] so subsequent commits resume normally instead of wiping progress again.

# ── Pull latest code from repo ──────────────────────────────────────────────────
import subprocess as _sp
_pull = _sp.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'],
                capture_output=True, text=True)
print("git pull:", _pull.stdout.strip() or "already up to date")

import os, time, pickle, traceback
from datetime import datetime
import pandas as pd
from tqdm.auto import tqdm

import importlib, sys
for mod in ['Mas_solver', 'baselines', 'evaluation_metrics']:
    if mod in sys.modules:
        del sys.modules[mod]
import Mas_solver
from Mas_solver import (
    QualityAwarePipeline, UnifiedLLMClient, AgentRole,
    HETEROGENEOUS_PRESETS, token_budget, _extract_last_number,
    SOLVER_VERSION,   # [v12.0] provenance stamp — checkpoints from other versions auto-discard
)
import baselines
from baselines import (
    direct_answer, chain_of_thought, self_consistency, baseline_only,
    pal, pot,                           # [v10.4] PAL + PoT strong baselines
    BaselineResult,
)

# ── WHICH SYSTEMS RUN THIS COMMIT (free-Kaggle split plan) ─────────────────
# Each phase = ONE "Save & Run All (Commit)", sized to fit the 12h limit.
# Run them in order, change RUN_PHASE each time, and download the CSV(s)
# from each commit's Output. Aggregate all CSVs at the end with
# aggregate_local.py (no GPU needed) or by collecting them into one folder.
#   'b1_direct'  -> just B1 baseline           (safest split)
#   'b2_cot'     -> just B2 (CoT) baseline
#   'b_pal'      -> just B-PAL baseline
#   'mas_full'   -> mas_sht_math7b             (full MAS-SHT; ~8h @ measured 197s/problem)
#   'mas_no_siv' -> mas_no_siv_math7b          (ablation; similar runtime to mas_full)
#   'baselines'  -> all 3 baselines in one commit (risk: may exceed 12h — prefer the split phases above)
#   'all'        -> everything (only if Cell 2.5 projects < ~11h)
# [v14.0] Only the two MAS phases need re-running: Cell 2's problem set is
# unchanged (same datasets, same seed=42), so b1_direct / b2_cot / b_pal from the
# v12.2 mix are still valid and comparable. SOLVER_VERSION is now 14.0, so old
# MAS checkpoints are auto-discarded by _load_ckpt -- leave RESET_SYSTEMS = [].
# Sequence for this round:  'mas_full'  ->  'mas_no_siv'   (one commit each)
RUN_PHASE = 'mas_full'    # <-- v12.2: Cell 2's dataset mix changed (weighted toward gsm-hard for
                          #     more genuine SIV coverage) -> this is now a DIFFERENT 150-problem set
                          #     than the pilot/v12.0/v12.1 runs, so b1/b2/b_pal must be re-run on it.
                          #     BEFORE this first commit: set RESET_SYSTEMS (below) to
                          #     ['b1_direct', 'b2_cot', 'b_pal', 'mas_sht_math7b'] for ONE run, then
                          #     back to []. Sequence: 'b1_direct' -> 'b2_cot' -> 'b_pal' -> 'mas_full'
                          #     -> 'mas_no_siv' (each a separate commit, ~150 problems/phase).

# ── [v14.0] Blueprint reasoning mode ──────────────────────────────────────
# True  -> Mathematician writes 'DERIVATION:' (free reasoning) then 'BLUEPRINT:'
#          (the JSON) in ONE call. Zero extra LLM calls.
# False -> v13.0 behaviour: JSON only, no scratchpad.
#
# Why this exists: on the v13.0 run the same model answered 100% of gsm8k_test
# correctly with free CoT while only 41.7% of its blueprints evaluated to the
# gold answer. The old prompt demanded "Return ONLY valid JSON, no preamble"
# AND "mentally trace through your equations" -- the model had nowhere to think.
#
# VALIDATE BEFORE COMMITTING 8 HOURS TO IT. Run the pre-test first (~15 min,
# Mathematician only, no Programmer/SHT/baselines):
#     !python {REPO_DIR}/pretest_blueprint_mode.py --n 20 --preset qwen_math7b_mixed
# It prints blueprint accuracy for both modes side by side. If reasoning-first
# does not clearly win, set this to False and keep the v13.0 behaviour -- an
# unvalidated prompt change does not belong in a thesis run.
#
# [RESULT, 2026-07-31, n=20, qwen_math7b_mixed] json_only 35% (7/20) vs
# reasoning_first 15% (3/20) blueprint-answer accuracy -- reasoning_first LOST,
# clearly, not within noise. It is also slower (larger token budget: 2304 vs
# 1536) and a pre-flight run under it projected an alarming ~27h for 150
# problems (partly inflated by one-time model-load time + a hard test problem
# triggering repair, but still a real cost increase with no accuracy benefit
# to show for it). Defaulting to False: keep the v13.0-proven prompt, and let
# the OTHER v14.0 changes (Layer 0 detection, repair-gate unlock, blueprint
# logging) be evaluated on their own without an unvalidated prompt change
# riding along and confounding the comparison.
MATH_REASONING_FIRST = False

_ALL_BASELINES = ['b1_direct', 'b2_cot', 'b_pal']
# MAS_PRESET comes from Cell 2.5's toggle (USE_MIXED_MATHEMATICIAN) -- falls
# back to the homogeneous preset if Cell 2.5 wasn't run yet this session.
MAS_PRESET = globals().get('MAS_PRESET', 'qwen_math_7b_local')
print(f"MAS_PRESET={MAS_PRESET!r}")
_MAS_FULL      = ('mas_sht_math7b',    MAS_PRESET, True,  True)   # SIV on,  SHT on
_MAS_NO_SIV    = ('mas_no_siv_math7b', MAS_PRESET, False, True)   # ablation: SIV off
_PHASES = {
    # Split per-baseline so EVERY commit is safely < 12h (measured MAS
    # avg was 197s/problem; a combined 3-baseline commit risked >12h).
    'b1_direct':  (['b1_direct'], []),
    'b2_cot':     (['b2_cot'],    []),
    'b_pal':      (['b_pal'],     []),
    'baselines':  (_ALL_BASELINES, []),   # combined — only if you're sure it fits 12h
    'mas_full':   ([], [_MAS_FULL]),
    'mas_no_siv': ([], [_MAS_NO_SIV]),
    'all':        (_ALL_BASELINES, [_MAS_FULL, _MAS_NO_SIV]),
}
if RUN_PHASE not in _PHASES:
    raise ValueError(f'RUN_PHASE={RUN_PHASE!r} not in {list(_PHASES)}')
_bl_run, _mas_run = _PHASES[RUN_PHASE]
print(f"RUN_PHASE={RUN_PHASE!r} -> baselines={_bl_run}, mas={[m[0] for m in _mas_run]}")

CONFIG = {
    'baselines_to_run': _bl_run,
    'baseline_client':  {'provider': 'local_hf',
                         'model':    'Qwen/Qwen2.5-Math-7B-Instruct',
                         'load_4bit': True},   # same model as MAS (4-bit) - fair comparison
    'mas_variants':     _mas_run,
    'inter_problem_delay': 0.5,
    'checkpoint_every': 5,
}

# ── Optional reset ───────────────────────────────────────────────────────────────────
# Set to [] to skip. List system names to delete their checkpoints + old CSVs.
RESET_SYSTEMS = []  # [] = RESUME from checkpoint (default!). To wipe a system from scratch, list it, e.g. ['b1_direct'].

import glob as _glob
for _sys in RESET_SYSTEMS:
    _ckpt = os.path.join(CHECKPOINT_DIR, f'{_sys}.pkl')
    if os.path.exists(_ckpt):
        os.remove(_ckpt)
        print(f"Deleted checkpoint: {_ckpt}")
    for _f in _glob.glob(os.path.join(RESULTS_DIR, f'{_sys}_*.csv')):
        os.remove(_f)
        print(f"Deleted CSV: {_f}")

# ── Helpers ──────────────────────────────────────────────────────────────────────────
TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')

def _ckpt_path(name): return os.path.join(CHECKPOINT_DIR, f'{name}.pkl')
def _csv_path(name):  return os.path.join(RESULTS_DIR,    f'{name}_{TIMESTAMP}.csv')

# [v14.1] Cross-commit resume. A Kaggle commit starts with an EMPTY /kaggle/working,
# so the .pkl checkpoint from a previous commit is gone and a run that hit the 12h
# wall had to restart from problem 0 -- which is why the same run kept dying at the
# same place. Attach the previous commit's CSV as a Kaggle Dataset (or leave it in
# any of the paths below) and its completed problems are skipped.
#   How: Kaggle -> Add Data -> your dataset containing e.g. mas_full_20260803.csv
RESUME_FROM_CSV = []   # e.g. ['/kaggle/input/mas-resume/mas_full_20260803.csv']

def _seed_rows_from_csv(name):
    """Rows for system `name` recovered from any attached CSV, else [].

    Matches on the CSV's own 'system' column, NOT on the filename: downloaded
    Kaggle outputs get renamed by hand all the time (mas_full_20260803.csv holds
    system='mas_sht_math7b'), and a filename-based match would silently find
    nothing and restart the run from zero -- the exact failure this is meant to
    prevent. Rows written by a different solver_version are refused, since the
    pipeline they came from is not the one about to continue.
    """
    import glob as _g
    paths = list(RESUME_FROM_CSV) or _g.glob('/kaggle/input/**/*.csv', recursive=True)
    best = []
    for p in paths:
        if not os.path.isfile(p):
            continue
        try:
            df = pd.read_csv(p)
        except Exception:
            continue
        if 'problem_id' not in df.columns or 'system' not in df.columns:
            continue
        df = df[df['system'].astype(str) == name]
        if df.empty:
            continue
        if 'solver_version' in df.columns and df['solver_version'].notna().any():
            v = str(df['solver_version'].dropna().iloc[0]).rstrip('0').rstrip('.')
            if v != str(SOLVER_VERSION).rstrip('0').rstrip('.'):
                print(f'  resume: SKIPPING {p} (solver_version={v!r} != {SOLVER_VERSION!r})')
                continue
        df = df.drop_duplicates(subset='problem_id', keep='last')
        if len(df) > len(best):
            best = df.to_dict('records')
            print(f'  resume: {len(best)} usable rows for {name} in {p}')
    if best:
        print(f'  resume: seeding {len(best)} completed problems for {name}')
    return best

def _load_ckpt(name, expected_preset=None):
    p = _ckpt_path(name)
    if os.path.isfile(p):
        try:
            with open(p, 'rb') as f:
                rows = pickle.load(f)
            # [v10.6] auto-discard checkpoints written by a different model/preset —
            # prevents stale rows (e.g. the invalid 1.5B pilot) polluting this run
            if rows and expected_preset and rows[0].get('preset') != expected_preset:
                print(f'  [{name}] stale checkpoint (preset={rows[0].get("preset")!r}, expected {expected_preset!r}) — starting fresh')
                return []
            # [v12.0] MAS checkpoints written by a different solver version mix
            # selection policies — discard them (baselines are unaffected).
            if rows and name.startswith('mas') and rows[0].get('solver_version') != SOLVER_VERSION:
                print(f'  [{name}] stale checkpoint (solver_version={rows[0].get("solver_version")!r}, '
                      f'expected {SOLVER_VERSION!r}) — starting fresh')
                return []
            return rows
        except Exception as e:
            print(f'  checkpoint load failed ({name}): {e}')
    return _seed_rows_from_csv(name)

def _save_ckpt(name, rows):
    with open(_ckpt_path(name), 'wb') as f:
        pickle.dump(rows, f)
    try:
        pd.DataFrame(rows).to_csv(_csv_path(name), index=False)  # live CSV every checkpoint -> half-runs are readable
    except Exception:
        pass

def _is_correct(pred, gold):
    """Numeric grader. Handles GSM8K '#### N' gold format."""
    if pred is None or gold is None:
        return False
    gold_s = str(gold).strip()
    if '####' in gold_s:
        gold_s = gold_s.split('####')[-1].strip()
    gold_s = gold_s.replace(',', '')
    try:
        gold_f = float(gold_s)
    except (ValueError, TypeError):
        return str(pred).strip() == gold_s
    pn = _extract_last_number(str(pred))
    if pn is None:
        return False
    return abs(float(pn) - gold_f) < 1e-3

def _row_from_baseline(system, preset, problem_id, gold, br: BaselineResult):
    return {
        'problem_id': problem_id, 'system': system, 'preset': preset,
        'gold': gold, 'predicted': br.answer,
        'correct': _is_correct(br.answer, gold),
        'baseline_ans': br.answer, 'baseline_correct': _is_correct(br.answer, gold),
        'mas_used_baseline_fallback': False, 'local_hf_fallback': False,
        'error_type': br.error_type,
        'time_s': br.time_s, 'num_llm_calls': br.num_llm_calls,
        'tokens_estimated': br.tokens_estimated,
        'verification_passed': None, 'verification_confidence': None,
        'solver_agent': system,
        'siv_execution_audit_passed': None, 'siv_blueprint_answer': None,
        'siv_execution_rel_error': None, 'siv_verified': None,
        'siv_confidence': None, 'siv_givens_matched': None,
        'siv_givens_total': None, 'siv_invertible': None,
        'siv_failed_givens': '[]', 'siv_unused_givens': '[]',
        'siv_undefined_symbols': '[]', 'siv_undeclared_given_keys': '[]',   # [v14.0]
        'siv_skipped': None, 'siv_tautological': None,                      # [v14.0]
        'blueprint_deterministic_fixes': '',                                 # [v14.2]
        'blueprint_provenance': '', 'blueprint_givens': '',                 # [v14.0]
        'blueprint_equations': '',
        'siv_verifies_translation': False,
        'sht_triggered': False, 'sht_triage': 'n/a',
        'sht_num_candidates': 0, 'sht_api_calls': 0,
        'solver_version': SOLVER_VERSION,
        'timestamp': datetime.now().isoformat(),
    }

def _row_from_mas(system, preset, problem_id, gold, mas_out, time_s):
    mas_block    = mas_out.get('mas', {}) or {}
    sht_block    = mas_out.get('sht', {}) or {}
    siv_block    = mas_out.get('siv', {}) or {}
    base_block   = mas_out.get('baseline', {}) or {}
    prog_metrics = mas_block.get('programmer_metrics', {}) or {}

    pred_str     = mas_block.get('answer', '')
    pred         = _extract_last_number(str(pred_str))
    base_ans_raw = base_block.get('answer', None)
    base_pred    = _extract_last_number(str(base_ans_raw)) if base_ans_raw is not None else None

    return {
        'problem_id': problem_id, 'system': system, 'preset': preset,
        'gold': gold, 'predicted': pred,
        'correct': _is_correct(pred_str, gold),  # [v10.5] grade RAW answer (symbolic-safe)
        'baseline_ans': base_pred, 'baseline_correct': _is_correct(base_ans_raw, gold),
        'mas_used_baseline_fallback': mas_block.get('used_baseline_fallback', False),
        'local_hf_fallback': mas_block.get('local_hf_fallback', False),
        'extracted_from_cot': mas_block.get('extracted_from_cot', False),  # [v11.0] did 2-step extraction engage the pipeline?
        'blueprint_repaired': mas_block.get('blueprint_repaired', False),  # [v13.0] did the audit-triggered repair loop fire?
        'error_type': '' if pred is not None else 'mas_returned_unknown',
        'time_s': time_s,
        'num_llm_calls': sht_block.get('api_calls_used', 3),
        'tokens_estimated': 0,
        'verification_passed': prog_metrics.get('verification_passed', True),
        'verification_confidence': prog_metrics.get('verification_confidence', 1.0),
        'solver_agent': (mas_out.get('agents', [None])[0].agent
                         if mas_out.get('agents') else 'unknown'),
        'siv_execution_audit_passed': siv_block.get('execution_audit_passed'),
        'siv_blueprint_answer': siv_block.get('blueprint_answer'),
        'siv_execution_rel_error': siv_block.get('execution_rel_error'),
        'siv_verified': siv_block.get('verified'),
        'siv_confidence': siv_block.get('confidence'),
        'siv_givens_matched': siv_block.get('givens_matched'),
        'siv_givens_total': siv_block.get('givens_total'),
        'siv_invertible': siv_block.get('invertible'),
        'siv_failed_givens': str(siv_block.get('failed_givens', [])),
        'siv_unused_givens': str(siv_block.get('unused_givens', [])),
        # [v14.0] Layer 0 — structural defects. 30/109 audited rows on the v13.0
        # run died here silently (blueprint_answer=None, invertible=False); these
        # two columns say WHICH name was undefined.
        'siv_undefined_symbols': str(siv_block.get('undefined_symbols', [])),
        'siv_undeclared_given_keys': str(siv_block.get('undeclared_given_keys', [])),
        # [v14.0] 'skipped' = SIV had nothing to audit (vs audited-and-failed);
        # 'tautological' = the audited answer came from the blueprint's own
        # equations, so Layer 1 passed by construction and proves nothing.
        # EXCLUDE tautological rows from any detector statistic.
        'siv_skipped': siv_block.get('skipped'),
        'siv_tautological': siv_block.get('tautological'),
        # [v14.0] The blueprint itself. Nothing about blueprint content was ever
        # recorded before, so blueprint quality could only be inferred from
        # siv_blueprint_answer and SIV could not be replayed offline.
        # [v14.2] deterministic (no-LLM) blueprint fixes applied, if any
        'blueprint_deterministic_fixes': mas_block.get('blueprint_deterministic_fixes', ''),
        'blueprint_provenance': mas_block.get('blueprint_provenance', ''),
        'blueprint_givens': mas_block.get('blueprint_givens', ''),
        'blueprint_equations': mas_block.get('blueprint_equations', ''),
        'siv_verifies_translation': False,
        # [v14.1] EVERY candidate's answer, flattened. solve() already builds
        # these (result['sht']['candidates']) but nothing ever wrote them out,
        # so the marginal accuracy of the Programmer/primary -- the single
        # number needed to decide what the do-no-harm invariant should anchor
        # to -- has never been measurable in ANY run. With these columns any
        # selection policy can be replayed offline from a finished CSV instead
        # of costing another 12h commit.
        **{f'cand_{c["id"]}': c.get('answer')
           for c in (sht_block.get('candidates') or []) if c.get('id')},
        'sht_final_strategy': sht_block.get('final_strategy', ''),
        'sht_triggered': sht_block.get('triggered', False),
        'sht_triage': sht_block.get('triage_result', 'n/a'),
        'sht_num_candidates': sht_block.get('num_candidates', 0),
        'sht_api_calls': sht_block.get('api_calls_used', 0),
        'solver_version': SOLVER_VERSION,
        'timestamp': datetime.now().isoformat(),
    }

# ── Run baselines ──────────────────────────────────────────────────────────────────────────
BASELINE_FNS = {
    'b1_direct':        direct_answer,
    'b2_cot':           chain_of_thought,
    'b3_sc5':           lambda c, p: self_consistency(c, p, n=5),
    'b4_baseline_only': baseline_only,
    'b_pal':            pal,        # [v10.4] PAL  (Gao et al., ICML 2023)
    'b_pot':            pot,        # [v10.4] PoT  (Chen et al., TMLR 2023)
}

if CONFIG['baselines_to_run']:
    bc = CONFIG['baseline_client']
    baseline_client = UnifiedLLMClient(
        provider=bc['provider'], model_override=bc['model'],
        load_4bit=bc.get('load_4bit', False))  # [v10.6] honor config — was hardcoded True: baseline ran 4-bit-handicapped vs fp16 MAS (unfair)
    for sys_name in CONFIG['baselines_to_run']:
        fn   = BASELINE_FNS[sys_name]
        rows = _load_ckpt(sys_name, expected_preset=bc['model'])
        done_ids = {r['problem_id'] for r in rows}
        print(f'\n[{sys_name}] resuming with {len(done_ids)}/{len(PROBLEMS)} done')
        for i, p in enumerate(tqdm(PROBLEMS, desc=sys_name)):
            if p['problem_id'] in done_ids:
                continue
            try:
                br = fn(baseline_client, p['question'])
            except Exception:
                br = BaselineResult(
                    answer=None, raw=traceback.format_exc()[-500:],
                    num_llm_calls=0, tokens_estimated=0,
                    time_s=0.0, error_type='exception')
            rows.append(_row_from_baseline(sys_name, bc['model'],
                                           p['problem_id'], p['gold_answer'], br))
            if (i + 1) % CONFIG['checkpoint_every'] == 0:
                _save_ckpt(sys_name, rows)
            time.sleep(CONFIG['inter_problem_delay'])
        _save_ckpt(sys_name, rows)
        df_b = pd.DataFrame(rows)
        df_b.to_csv(_csv_path(sys_name), index=False)
        n_ok = df_b['correct'].sum()
        print(f'[{sys_name}] done — {n_ok}/{len(df_b)} correct ({n_ok/len(df_b)*100:.1f}%)')

# ── VRAM cleanup helper ─────────────────────────────────────────────────────────────
import gc

def _free_local_hf_models(pipeline_obj):
    freed = 0
    for key, client in list(getattr(pipeline_obj, '_client_cache', {}).items()):
        mdl = getattr(client, '_local_model', None)
        if mdl is not None:
            try:
                import torch; mdl.cpu(); del mdl
            except Exception:
                pass
            client._local_model = None
            client._local_tokenizer = None
            freed += 1
    if freed:
        gc.collect()
        try:
            import torch
            torch.cuda.empty_cache()
        except Exception:
            pass
        print(f"  [VRAM] freed {freed} local_hf model(s) — GPU cache cleared")

# ── Free baseline VRAM before MAS variants ───────────────────────────────────────
if CONFIG['baselines_to_run']:
    _bl_mdl = getattr(baseline_client, '_local_model', None)
    if _bl_mdl is not None:
        try:
            _bl_mdl.cpu(); del _bl_mdl
        except Exception:
            pass
        baseline_client._local_model = None
        baseline_client._local_tokenizer = None
        gc.collect()
        torch.cuda.empty_cache()
        print('[VRAM] baseline model freed before MAS variants')

# ── Run MAS variants ──────────────────────────────────────────────────────────────────────────
for sys_name, preset, en_siv, en_sht in CONFIG['mas_variants']:
    rows = _load_ckpt(sys_name, expected_preset=preset)
    done_ids = {r['problem_id'] for r in rows}
    print(f'\n[{sys_name}] preset={preset} siv={en_siv} sht={en_sht} '
          f'resuming with {len(done_ids)}/{len(PROBLEMS)} done')
    pipeline = QualityAwarePipeline(
        heterogeneous_preset=preset, use_cache=False,
        enable_siv=en_siv, enable_sht=en_sht,
        evaluation_mode=True,           # [v10.4] forces cache off during evaluation
        dataset_seed=42,                # [v10.4] reproducibility
        math_reasoning_first=MATH_REASONING_FIRST,   # [v14.0] see toggle above
    )
    # [v14.0] Fail-fast sample size + counters. Cell 2.5's pre-flight guard uses
    # its OWN pipeline instance on 2 problems and frees it before this one loads
    # -- it can pass while THIS reload still fails (e.g. VRAM fragmentation from
    # the first load, or a second model competing for the same GPU under the
    # mixed preset). On 2026-07-31 that gap let a commit run 150/150 problems to
    # an all-empty blueprint (functionally baseline-only) before anyone noticed.
    # Checking a small, cheap sample of FRESH problems here closes it.
    _FAILFAST_N = 5
    _fresh_seen = 0
    _fresh_empty = 0
    for i, p in enumerate(tqdm(PROBLEMS, desc=sys_name)):
        if p['problem_id'] in done_ids:
            continue
        t0 = time.time()
        try:
            mas_out = pipeline.solver.solve(p['question'], str(p['gold_answer']))
        except Exception as e:
            print(f'  exception on {p["problem_id"]}: {e}')
            mas_out = {'mas': {'answer': 'unknown'}, 'siv': {}, 'sht': {}}
        elapsed = time.time() - t0
        row = _row_from_mas(sys_name, preset,
                            p['problem_id'], p['gold_answer'], mas_out, elapsed)
        rows.append(row)

        _fresh_seen += 1
        if row.get('blueprint_givens') in ('{}', '', None):
            _fresh_empty += 1
        if _fresh_seen == _FAILFAST_N and _fresh_empty == _fresh_seen:
            _save_ckpt(sys_name, rows)
            raise RuntimeError(
                f"[{sys_name}] FAIL-FAST: the first {_FAILFAST_N} fresh problems ALL "
                f"came back with an empty blueprint (blueprint_givens='{{}}'). The "
                f"Mathematician is almost certainly erroring on every call -- check "
                f"the log above for 'ERROR_GENERATION' / 'OutOfMemoryError'. Aborting "
                f"before burning the rest of the run; completed rows are checkpointed "
                f"and will be skipped on resume once the underlying issue is fixed."
            )

        if (i + 1) % CONFIG['checkpoint_every'] == 0:
            _save_ckpt(sys_name, rows)
        time.sleep(CONFIG['inter_problem_delay'])
    _save_ckpt(sys_name, rows)
    df_m = pd.DataFrame(rows)
    df_m.to_csv(_csv_path(sys_name), index=False)
    n_ok = df_m['correct'].sum()
    print(f'[{sys_name}] done — {n_ok}/{len(df_m)} correct ({n_ok/len(df_m)*100:.1f}%)')
    _free_local_hf_models(pipeline)

print('\n=== All runs complete ===')

## Cell 4 — Aggregation

In [ ]:
# === Cell 4 — Aggregation ======================================================
# Merges all CSVs in RESULTS_DIR, deduplicates by (problem_id, system),
# and prints a summary table. Run after Cell 3 finishes.

import glob, os
import pandas as pd

csv_paths = sorted(glob.glob(os.path.join(RESULTS_DIR, '*.csv')))
print(f'Found {len(csv_paths)} CSV(s):')
for p in csv_paths:
    print(f'  {os.path.basename(p)}')

if not csv_paths:
    raise SystemExit('No results yet — run Cell 3 first.')

frames = [pd.read_csv(p) for p in csv_paths]
merged = pd.concat(frames, ignore_index=True)
merged['timestamp'] = pd.to_datetime(merged.get('timestamp'), errors='coerce')
merged = (merged
          .sort_values('timestamp')
          .drop_duplicates(['problem_id', 'system'], keep='last')
          .reset_index(drop=True))

# [v10.6] Keep only rows from the CURRENT experiment — drops stale CSVs
# from older sessions (e.g. the invalid 1.5B / 4-bit pilot runs).
EXPECTED_PRESETS = {'qwen_math_7b_local', 'qwen_math7b_mixed',
                     'qwen_math7b_mixed_mathematician_instruct',
                     'Qwen/Qwen2.5-Math-7B-Instruct'}  # local run: MAS preset(s) + baseline model id
if 'preset' in merged.columns:
    _n0 = len(merged)
    merged = merged[merged['preset'].isin(EXPECTED_PRESETS)].reset_index(drop=True)
    if _n0 - len(merged):
        print(f'[filter] dropped {_n0 - len(merged)} stale row(s) from older runs')

results_dict = {sys: g.reset_index(drop=True)
                for sys, g in merged.groupby('system')}

summary = (merged
           .groupby('system')
           .agg(n=('problem_id', 'nunique'),
                accuracy=('correct', 'mean'),
                avg_time_s=('time_s', 'mean'),
                avg_llm_calls=('num_llm_calls', 'mean'))
           .sort_values('accuracy', ascending=False)
           .round(4))
summary['accuracy_pct'] = (summary['accuracy'] * 100).round(1).astype(str) + '%'

print('\n=== Summary ===')
print(summary[['n', 'accuracy_pct', 'avg_time_s', 'avg_llm_calls']].to_string())

# Also check if MAS csv was already produced in a previous session
if 'mas_sht_math7b' in results_dict:
    df = results_dict['mas_sht_math7b']
    sht = df['sht_triggered'].sum()
    siv = (df['siv_verified'] == True).sum()
    print(f'\nMAS extras: SHT triggered={sht}/{len(df)}, SIV verified={siv}/{len(df)}')


## Cell 5 — Statistical Analysis

Wilson 95 % CI per system, paired bootstrap CI on Δ-accuracy, McNemar test.  Reads from `evaluation_metrics.py` (v10.4) — no manual scipy needed.

In [ ]:
# === Cell 5 — Statistical Analysis ============================================
# Uses evaluation_metrics.py (v10.4):
#   • Wilson 95 % CI on per-system accuracy    → accuracy_ci_lo / accuracy_ci_hi
#   • Paired bootstrap 95 % CI on Δ-accuracy     → delta_ci_lo / delta_ci_hi
#   • McNemar test (exact or Yates-corrected)  → p_value / significant_at_alpha
# Saves artifacts/comparison_table.{csv,md} and artifacts/mcnemar_results.csv

import sys, os
sys.path.insert(0, REPO_DIR)
import importlib
for _m in ['evaluation_metrics']:
    if _m in sys.modules:
        del sys.modules[_m]
from evaluation_metrics import compute_all_metrics, run_mcnemar_tests

# ── Auto-detect reference system ────────────────────────────────────────────────────────
_mas_systems = [s for s in results_dict if s.startswith('mas_sht_')]
if not _mas_systems:
    print('No mas_sht_* system in results yet - skipping stats (normal for a baseline-only commit).')
else:
    REFERENCE = 'mas_sht_math7b' if 'mas_sht_math7b' in _mas_systems else _mas_systems[0]
    print(f'Reference system : {REFERENCE}')
    print(f'All systems      : {sorted(results_dict.keys())}')

    # ── Full metrics table (accuracy + Wilson CI + delta + bootstrap CI) ──────────
    metrics_df = compute_all_metrics(results_dict, reference_system=REFERENCE)
    comp_csv   = os.path.join(ARTIFACTS_DIR, 'comparison_table.csv')
    metrics_df.to_csv(comp_csv, index=False)
    print(f'\nSaved: {comp_csv}')

    _cols = ['system', 'n_paired', 'accuracy', 'accuracy_ci_lo', 'accuracy_ci_hi',
             'delta_vs_ref', 'delta_ci_lo', 'delta_ci_hi',
             'avg_llm_calls', 'avg_time_s', 'accuracy_per_call']
    print('\n=== Comparison table ===')
    print(metrics_df[[c for c in _cols if c in metrics_df.columns]].to_string(index=False))

    # ── McNemar tests ──────────────────────────────────────────────────────────────────────────
    mcnemar_df = run_mcnemar_tests(results_dict, reference_system=REFERENCE)
    mc_csv     = os.path.join(ARTIFACTS_DIR, 'mcnemar_results.csv')
    mcnemar_df.to_csv(mc_csv, index=False)
    print(f'\nSaved: {mc_csv}')

    _mc_cols = ['other_system', 'n_paired', 'a', 'b', 'c', 'd',
                'test_used', 'p_value', 'significant_at_alpha']
    print('\n=== McNemar results ===')
    print(mcnemar_df[[c for c in _mc_cols if c in mcnemar_df.columns]].to_string(index=False))

    # ── Markdown table for copy-paste into thesis ─────────────────────────────────────
    try:
        from tabulate import tabulate
        _display = ['system', 'n_paired', 'accuracy', 'accuracy_ci_lo', 'accuracy_ci_hi',
                    'delta_vs_ref', 'delta_ci_lo', 'delta_ci_hi', 'avg_llm_calls']
        md_table = tabulate(
            metrics_df[[c for c in _display if c in metrics_df.columns]],
            headers='keys', tablefmt='pipe', floatfmt='.3f', showindex=False,
        )
        md_path = os.path.join(ARTIFACTS_DIR, 'comparison_table.md')
        with open(md_path, 'w') as f:
            f.write(md_table)
        print(f'\nSaved: {md_path}')
        print(md_table)
    except ImportError:
        print('tabulate not installed — skipping Markdown table')

## Cell 5.5 — SIV Evaluation (novel contribution)


In [ ]:
# === Cell 5.5 — SIV Evaluation ================================================
# Turns the per-problem SIV columns into the thesis tables using siv_evaluation.py:
#   1. Coverage funnel  — how often the inspector actually had a blueprint to check
#   2. Detector metrics — does the inspector's verdict track answer correctness?
#   3. Error-layer split — execution errors (inspector catches) vs translation
#                          errors (NL->math, inspector is blind by design)
#   4. Confident-skip efficacy — does it save API calls without losing accuracy?
#   5. Ablation (only if a 'mas_no_siv' run exists) — inspector ON vs OFF
#
# Saves artifacts/siv_evaluation_report.md + one CSV per table.

import sys, os, importlib
sys.path.insert(0, REPO_DIR)
for _m in ['siv_evaluation']:
    if _m in sys.modules:
        del sys.modules[_m]
import siv_evaluation as se

_mas = [s for s in results_dict if s.startswith('mas_sht_')]
if not _mas:
    print('No mas_sht_* system in results yet - skipping SIV eval (normal for a baseline-only commit).')
else:
    FULL   = 'mas_sht_math7b' if 'mas_sht_math7b' in _mas else _mas[0]
    NO_SIV = next((s for s in results_dict if 'no_siv' in s), None)

    full_df   = results_dict[FULL]
    no_siv_df = results_dict[NO_SIV] if NO_SIV else None
    print(f'SIV system : {FULL}')
    print(f'No-SIV run : {NO_SIV or "(none — ablation section skipped; add a mas_no_siv variant in Cell 3 to populate it)"}')

    # --- Full markdown report (saved + printed) ---
    report = se.siv_report(full_df, no_siv_df)
    _out = os.path.join(ARTIFACTS_DIR, 'siv_evaluation_report.md')
    with open(_out, 'w', encoding='utf-8') as f:
        f.write(report)
    print('\nSaved:', _out)
    print('\n' + '=' * 72)
    print(report)

    # --- Each table as its own CSV for direct import into the thesis ---
    se.siv_coverage(full_df).to_csv(os.path.join(ARTIFACTS_DIR, 'siv_coverage.csv'), index=False)
    se.siv_detector_metrics(full_df).to_csv(os.path.join(ARTIFACTS_DIR, 'siv_detector.csv'), index=False)
    se.error_layer_decomposition(full_df).to_csv(os.path.join(ARTIFACTS_DIR, 'siv_error_layers.csv'), index=False)
    se.siv_skip_efficacy(full_df).to_csv(os.path.join(ARTIFACTS_DIR, 'siv_skip_efficacy.csv'), index=False)
    if no_siv_df is not None:
        se.siv_ablation(full_df, no_siv_df).to_csv(os.path.join(ARTIFACTS_DIR, 'siv_ablation.csv'), index=False)
    print('\nSIV CSV tables written to', ARTIFACTS_DIR)


## Cell 6 — Plots

In [ ]:
# === Cell 6 — Plots ============================================================
# Accuracy bar chart (all systems) + SHT/SIV breakdown for MAS.
# Saves PNG to ARTIFACTS_DIR so you can download it.

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import os

COLORS = {
    'b1_direct':        '#8ecae6',
    'b2_cot':           '#219ebc',
    'b4_baseline_only': '#023047',
    'b_pal':            '#6a4c93',   # [v10.4] PAL
    'b_pot':            '#9b5de5',   # [v10.4] PoT
    'mas_sht_qwen3_32b': '#e76f51',
    'mas_sht_qwen7b':   '#f4845f',
    'mas_no_siv_qwen7b':'#fca17d',
    'mas_no_sht_qwen7b':'#fcbf49',
    'mas_sht_tiny':     '#e76f51',
    'mas_sht_deepseek7b':'#c1440e',
    'mas_sht_math7b':    '#e76f51',
    'mas_no_siv_math7b': '#fca17d',
}
LABELS = {
    'b1_direct':        'B1: Direct',
    'b2_cot':           'B2: CoT',
    'b4_baseline_only': 'B4: Baseline\nOnly',
    'b_pal':            'B-PAL',     # [v10.4]
    'b_pot':            'B-PoT',     # [v10.4]
    'mas_sht_qwen7b':   'MAS-SHT\nQwen-7B',
    'mas_no_siv_qwen7b':'MAS no-SIV\nQwen-7B',
    'mas_no_sht_qwen7b':'MAS no-SHT\nQwen-7B',
    'mas_sht_tiny':     'MAS-SHT\n1.5B',
    'mas_sht_deepseek7b':'MAS-SHT\nDS-7B',
    'mas_sht_math7b':    'MAS-SHT\nQwen-Math-7B',
    'mas_no_siv_math7b': 'MAS no-SIV\nQwen-Math-7B',
}

# Preferred display order — baselines first, then MAS variants
_DISPLAY_ORDER = [
    'b1_direct', 'b2_cot', 'b4_baseline_only', 'b_pal', 'b_pot',
    'mas_sht_tiny', 'mas_sht_qwen3_32b', 'mas_sht_qwen7b', 'mas_sht_deepseek7b',
    'mas_no_siv_qwen7b', 'mas_no_sht_qwen7b',
    'mas_sht_math7b', 'mas_no_siv_math7b',
]
systems = [s for s in _DISPLAY_ORDER if s in results_dict]
# Add any systems not in the preferred order at the end
systems += [s for s in sorted(results_dict.keys()) if s not in systems]
accs = [results_dict[s]['correct'].mean() for s in systems]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
n_sys = len(systems)
fig.suptitle(f'MAS-SHT v10.4 — GSM8K Results (n={len(PROBLEMS)}, {n_sys} systems)',
             fontsize=13, y=1.01)

# Panel 1: accuracy bars
ax = axes[0]
x_labels = [LABELS.get(s, s) for s in systems]
bars = ax.bar(x_labels,
              [a * 100 for a in accs],
              color=[COLORS.get(s, '#999') for s in systems],
              edgecolor='white', linewidth=1.2, width=0.5)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.8,
            f'{acc*100:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_ylabel('Accuracy (%)', fontsize=11)
ax.set_title('System Accuracy Comparison', fontsize=11)
ax.set_ylim(0, 110)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(axis='x', labelsize=8, rotation=15)

# Panel 2: MAS-SHT breakdown (SHT / SIV)
ax2 = axes[1]
_ref = REFERENCE if 'REFERENCE' in dir() and REFERENCE in results_dict else \
       next((s for s in _DISPLAY_ORDER if s in results_dict and s.startswith('mas_sht_')), None)
if _ref:
    df_m   = results_dict[_ref]
    total  = len(df_m)
    n_sht  = df_m['sht_triggered'].sum()
    n_nsht = total - n_sht
    sht_acc  = df_m[df_m['sht_triggered'] == True ]['correct'].mean() if n_sht  else 0.0
    nsht_acc = df_m[df_m['sht_triggered'] == False]['correct'].mean() if n_nsht else 0.0
    n_siv_t  = (df_m['siv_verified'] == True).sum()
    n_siv_f  = (df_m['siv_verified'] == False).sum()
    n_siv_na = df_m['siv_verified'].isna().sum()

    cats   = ['No SHT\n(conf. pass)', 'SHT\n(triggered)']
    vals   = [nsht_acc * 100, sht_acc * 100]
    counts = [n_nsht, n_sht]
    colors = ['#2a9d8f', '#e9c46a']
    brs = ax2.bar(cats, vals, color=colors, edgecolor='white', linewidth=1.2, width=0.4)
    for bar, acc, cnt in zip(brs, vals, counts):
        ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.8,
                 f'{acc:.1f}%\n(n={cnt})', ha='center', va='bottom', fontsize=10)
    ax2.set_ylabel('Accuracy (%)', fontsize=11)
    ax2.set_title(f'{_ref}: Accuracy by SHT Path\n'
                  f'SIV: verified={n_siv_t}, not-inv.={n_siv_f}, N/A={n_siv_na}',
                  fontsize=10)
    ax2.set_ylim(0, 110)
    ax2.spines['top'].set_visible(False)
    ax2.spines['right'].set_visible(False)
else:
    ax2.text(0.5, 0.5, 'MAS data not available', ha='center', va='center',
             transform=ax2.transAxes)

plt.tight_layout()
fig_path = os.path.join(ARTIFACTS_DIR, 'comparison.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')
print(f'\nTo download all results: File Explorer (left panel) → /kaggle/working/MAS_SHT/')